<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/IDB/W10-DEN/Q2/ASSIGNMENT_2.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# TASK 2: CONFIGURE DATABASE CHECKPOINTS AND ACTIVE LISTS
# Google Colab - Complete Assignment Solution
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

os.chdir("/content")

In [3]:
# ============================================================
# STEP 1: INSTALL REQUIRED LIBRARY
# ============================================================

try:
    import docx
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "python-docx"]
    )

from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

In [4]:
# ============================================================
# STEP 2: CREATE database_checkpoint.py
# ============================================================

python_code = r'''
from pathlib import Path


def get_active_transactions(log_file, upto_record=None):
    """
    Return the list of active transactions at a given
    point in the transaction log.

    A transaction is active if it has started but has
    neither committed nor aborted.
    """

    active = set()

    with open(log_file, "r") as file:
        records = file.readlines()

    if upto_record is not None:
        records = records[:upto_record]

    for record in records:

        record = record.strip()

        if not record:
            continue

        if record.startswith("<CHECKPOINT"):
            continue

        parts = record.strip("<>").split()

        if len(parts) < 2:
            continue

        transaction_id = parts[0]
        operation = parts[1].upper()

        if operation == "START":
            active.add(transaction_id)

        elif operation in ("COMMIT", "ABORT"):
            active.discard(transaction_id)

    return sorted(active)


class DatabaseCheckpoint:

    def __init__(self, filename):
        self.filename = Path(filename)

        # Volatile memory buffer
        self.volatile_buffer = []

        # Simulated stable storage
        self.stable_storage = []

    def add_log(self, record):
        """Add a log record to volatile memory."""

        self.volatile_buffer.append(record)
        print(f"[BUFFER] {record}")

    def flush_to_stable_storage(self):
        """
        Flush all records from volatile memory
        into simulated stable storage.
        """

        self.stable_storage.extend(self.volatile_buffer)

        self.volatile_buffer.clear()

        with open(self.filename, "w") as file:
            for record in self.stable_storage:
                file.write(record + "\n")

        print("[FLUSH] Volatile records written to stable storage.")

    def create_checkpoint(self):
        """
        Flush pending records and create a checkpoint
        containing all currently active transactions.
        """

        # First flush all previous log records
        self.flush_to_stable_storage()

        # Determine active transactions
        active = get_active_transactions(self.filename)

        active_list = ", ".join(active)

        if not active_list:
            active_list = "NONE"

        checkpoint_record = (
            f"<CHECKPOINT L: {active_list}>"
        )

        print("\n[CHECKPOINT CREATED]")
        print("Active transactions:", active)

        # Add checkpoint to stable storage
        self.volatile_buffer.append(checkpoint_record)

        # Flush checkpoint record too
        self.flush_to_stable_storage()

        return active


def generate_journal(filename):
    """Generate a sample transaction journal."""

    db = DatabaseCheckpoint(filename)

    print("\nGENERATING TRANSACTION LOGS")
    print("=" * 55)

    # Transactions before checkpoint
    db.add_log("<T1 START>")
    db.add_log("<T1 UPDATE X 100 150>")
    db.add_log("<T2 START>")
    db.add_log("<T1 COMMIT>")
    db.add_log("<T2 UPDATE Y 200 250>")
    db.add_log("<T3 START>")
    db.add_log("<T3 UPDATE Z 300 350>")

    # Create checkpoint
    active = db.create_checkpoint()

    # Transactions after checkpoint
    db.add_log("<T3 COMMIT>")
    db.add_log("<T2 UPDATE Y 250 275>")
    db.add_log("<T4 START>")
    db.add_log("<T4 UPDATE W 400 450>")
    db.add_log("<T2 COMMIT>")
    db.add_log("<T4 ABORT>")

    # Flush remaining records
    db.flush_to_stable_storage()

    return active


def recovery_simulation(log_file):
    """
    Compare recovery scanning without checkpointing
    and recovery scanning from the latest checkpoint.
    """

    with open(log_file, "r") as file:
        records = [
            line.strip()
            for line in file
            if line.strip()
        ]

    total_records = len(records)

    # Find most recent checkpoint
    checkpoint_index = None

    for index in range(len(records) - 1, -1, -1):

        if records[index].startswith("<CHECKPOINT"):
            checkpoint_index = index
            break

    if checkpoint_index is None:
        raise ValueError("No checkpoint found in journal.")

    checkpoint_record = records[checkpoint_index]

    print("\n" + "=" * 55)
    print("DATABASE RECOVERY SIMULATION")
    print("=" * 55)

    print("\nLatest checkpoint:")
    print(checkpoint_record)

    # Extract active transaction list
    active_text = checkpoint_record.split(":", 1)[1]
    active_text = active_text.rstrip(">").strip()

    if active_text == "NONE":
        checkpoint_active = []
    else:
        checkpoint_active = [
            item.strip()
            for item in active_text.split(",")
        ]

    print("Transactions active at checkpoint:",
          checkpoint_active)

    # Without checkpoint: scan entire log
    without_checkpoint = total_records

    # With checkpoint: scan checkpoint and subsequent records
    scanned_records = records[checkpoint_index:]

    with_checkpoint = len(scanned_records)

    print("\nRECOVERY WITHOUT CHECKPOINT:")
    print("Records scanned:", without_checkpoint)

    print("\nRECOVERY WITH CHECKPOINT:")
    print("Records scanned:", with_checkpoint)

    # Display records scanned after checkpoint
    print("\nRecords scanned from latest checkpoint:")

    for record in scanned_records:
        print(record)

    # Recovery analysis: start with checkpoint active list
    # and process subsequent records.
    active = set(checkpoint_active)

    for record in scanned_records[1:]:

        parts = record.strip("<>").split()

        if len(parts) < 2:
            continue

        transaction_id = parts[0]
        operation = parts[1].upper()

        if operation == "START":
            active.add(transaction_id)

        elif operation in ("COMMIT", "ABORT"):
            active.discard(transaction_id)

    print("\nACTIVE TRANSACTIONS AFTER RECOVERY ANALYSIS:")
    print(sorted(active))

    improvement = (
        (without_checkpoint - with_checkpoint)
        / without_checkpoint
    ) * 100

    print("\nRECOVERY EFFICIENCY:")
    print(f"Records saved: {without_checkpoint - with_checkpoint}")
    print(f"Scan reduction: {improvement:.2f}%")

    print("\nCHECKPOINT RECOVERY COMPLETED SUCCESSFULLY.")

    return {
        "without_checkpoint": without_checkpoint,
        "with_checkpoint": with_checkpoint,
        "saved": without_checkpoint - with_checkpoint,
        "reduction": improvement,
        "checkpoint_active": checkpoint_active
    }


if __name__ == "__main__":

    log_file = Path(__file__).parent / "database_journal.log"

    generate_journal(log_file)

    print("\nACTIVE TRANSACTIONS AT CHECKPOINT:")

    # The checkpoint is on line 8 (1-based).
    active = get_active_transactions(
        log_file,
        upto_record=7
    )

    print(active)

    recovery_simulation(log_file)
'''

with open("database_checkpoint.py", "w") as file:
    file.write(python_code)

In [5]:
# ============================================================
# STEP 3: EXECUTE PYTHON SCRIPT
# ============================================================

print("\nEXECUTING DATABASE CHECKPOINT PROGRAM")
print("=" * 55)

result = subprocess.run(
    [sys.executable, "database_checkpoint.py"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Checkpoint simulation failed.")

print(result.stdout)


EXECUTING DATABASE CHECKPOINT PROGRAM

GENERATING TRANSACTION LOGS
[BUFFER] <T1 START>
[BUFFER] <T1 UPDATE X 100 150>
[BUFFER] <T2 START>
[BUFFER] <T1 COMMIT>
[BUFFER] <T2 UPDATE Y 200 250>
[BUFFER] <T3 START>
[BUFFER] <T3 UPDATE Z 300 350>
[FLUSH] Volatile records written to stable storage.

[CHECKPOINT CREATED]
Active transactions: ['T2', 'T3']
[FLUSH] Volatile records written to stable storage.
[BUFFER] <T3 COMMIT>
[BUFFER] <T2 UPDATE Y 250 275>
[BUFFER] <T4 START>
[BUFFER] <T4 UPDATE W 400 450>
[BUFFER] <T2 COMMIT>
[BUFFER] <T4 ABORT>
[FLUSH] Volatile records written to stable storage.

ACTIVE TRANSACTIONS AT CHECKPOINT:
['T2', 'T3']

DATABASE RECOVERY SIMULATION

Latest checkpoint:
<CHECKPOINT L: T2, T3>
Transactions active at checkpoint: ['T2', 'T3']

RECOVERY WITHOUT CHECKPOINT:
Records scanned: 14

RECOVERY WITH CHECKPOINT:
Records scanned: 7

Records scanned from latest checkpoint:
<CHECKPOINT L: T2, T3>
<T3 COMMIT>
<T2 UPDATE Y 250 275>
<T4 START>
<T4 UPDATE W 400 450>
<T2 C

In [6]:
# ============================================================
# STEP 4: READ RESULTS FOR WORD DOCUMENT
# ============================================================

# The generated journal contains 14 records.
with open("database_journal.log", "r") as file:
    journal_records = [
        line.strip()
        for line in file
        if line.strip()
    ]

total_records = len(journal_records)

checkpoint_index = next(
    i for i in range(len(journal_records) - 1, -1, -1)
    if journal_records[i].startswith("<CHECKPOINT")
)

without_checkpoint = total_records
with_checkpoint = total_records - checkpoint_index

records_saved = without_checkpoint - with_checkpoint

reduction = (
    records_saved / without_checkpoint
) * 100


In [7]:
# ============================================================
# STEP 5: CREATE recovery_efficiency.docx
# ============================================================

document = Document()

title = document.add_heading(
    "TASK 2: DATABASE CHECKPOINTING AND RECOVERY EFFICIENCY",
    level=1
)

title.alignment = WD_ALIGN_PARAGRAPH.CENTER

document.add_paragraph(
    "Objective: To simulate database checkpointing, "
    "identify active transactions, flush log records "
    "to stable storage, and compare recovery efficiency."
)

document.add_heading("1. Checkpoint Information", level=2)

checkpoint_record = journal_records[checkpoint_index]

document.add_paragraph(
    f"Checkpoint record: {checkpoint_record}"
)

document.add_paragraph(
    "Active transactions at checkpoint: T2, T3"
)

document.add_heading("2. Recovery Efficiency Comparison", level=2)

table = document.add_table(rows=1, cols=3)

table.style = "Table Grid"
table.alignment = WD_TABLE_ALIGNMENT.CENTER

headers = table.rows[0].cells

headers[0].text = "Parameter"
headers[1].text = "Without Checkpoint"
headers[2].text = "With Checkpoint"

comparison = [
    ("Starting point", "Beginning of log", "Latest checkpoint"),
    ("Log records scanned", str(without_checkpoint),
     str(with_checkpoint)),
    ("Records saved", "0", str(records_saved)),
    ("Scan reduction", "0%",
     f"{reduction:.2f}%")
]

for row_data in comparison:
    cells = table.add_row().cells

    for i, value in enumerate(row_data):
        cells[i].text = value

document.add_heading("3. Explanation", level=2)

document.add_paragraph(
    "A database checkpoint records the active transactions "
    "at a particular point in time. Before creating the "
    "checkpoint, pending log records are flushed from volatile "
    "memory to simulated stable storage. The checkpoint record "
    "is also written to stable storage."
)

document.add_paragraph(
    "During recovery, the system uses the latest checkpoint "
    "as its starting point instead of scanning the entire log. "
    "The checkpoint's active transaction list provides the "
    "initial transaction state. Subsequent records are analyzed "
    "to determine which transactions committed, aborted, or "
    "remain incomplete."
)

document.add_paragraph(
    "Therefore, checkpointing reduces the number of log records "
    "that must be examined during recovery and improves recovery "
    "efficiency."
)

document.add_heading("4. Conclusion", level=2)

document.add_paragraph(
    f"The simulation examined {without_checkpoint} log records "
    f"without checkpointing and {with_checkpoint} records "
    f"with checkpointing. It saved {records_saved} record scans, "
    f"representing a {reduction:.2f}% reduction."
)

# Improve document font
for paragraph in document.paragraphs:
    for run in paragraph.runs:
        run.font.name = "Arial"
        run.font.size = Pt(10)

document.save("recovery_efficiency.docx")

In [8]:
# ============================================================
# STEP 6: FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 55)
print("ALL TASK 2 FILES GENERATED SUCCESSFULLY")
print("=" * 55)

for filename in [
    "database_checkpoint.py",
    "database_journal.log",
    "recovery_efficiency.docx"
]:
    path = Path("/content") / filename
    print(f"[SAVED] {path} | {path.stat().st_size} bytes")

print("\nOpen the Files panel in Google Colab to access your files.")


ALL TASK 2 FILES GENERATED SUCCESSFULLY
[SAVED] /content/database_checkpoint.py | 6872 bytes
[SAVED] /content/database_journal.log | 224 bytes
[SAVED] /content/recovery_efficiency.docx | 37498 bytes

Open the Files panel in Google Colab to access your files.
